In [7]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from script.preprocessing import make_preprocessor, drop_feature_extremes, split_xy
from script.modeling import make_model, NUMERIC_FEATURES, CATEGORICAL_FEATURES


from script.config import TARGET, MAX_FLOORS, SECOND_TARGET
from script.run_cleaning import clean_dataset, save_datasets
from script.clean_rows import audit_rows
from script.clean_columns import drop_leakage_columns
from script.features import (add_age_features, add_use_features, add_surface_features,
                             add_location_features, build_features, NEW_FEATURES)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Chemin vers le fichier de données fourni
DATA_PATH = "./../data/2016-building-energy-benchmarking.csv"


In [8]:
building_consumption = pd.read_csv(DATA_PATH)
print("Shape brut :", building_consumption.shape)

# Diagnostic des incohérences AVANT nettoyage (ne modifie rien)
display(audit_rows(building_consumption))

# Nettoyage complet (script/run_cleaning.py) : filtres de lignes + traitement des colonnes
df, log = clean_dataset(building_consumption)


Shape brut : (3376, 46)


,nb de lignes
Cible manquante ou <= 0,23
Surface totale <= 0,0
GFA total != Parking + Bâtiment,0
NumberofFloors == 0,16
NumberofFloors > 80,1
NumberofBuildings == 0,92
"YearBuilt hors [1800, 2016]",0
"Surface du plus grand usage > 1,2 x surface totale",86
"Somme des 3 usages > 1,05 x surface totale",503
Plus grand usage manquant,20


Exclusion Multifamily (BuildingType) : 3376 -> 1668 (-1708 lignes, -50.6% vs précédent, 49.4% du brut restant)
Exclusion Multifamily (colonnes d'usage) : 1668 -> 1629 (-39 lignes, -2.3% vs précédent, 48.3% du brut restant)
Conservation des seuls Compliant : 1629 -> 1510 (-119 lignes, -7.3% vs précédent, 44.7% du brut restant)
Suppression cible manquante ou <= 0 : 1510 -> 1510 (-0 lignes, -0.0% vs précédent, 44.7% du brut restant)
NumberofBuildings = 0 -> NaN + BuildingsMissing : 50 lignes concernées (3.3% du jeu), conservées
NumberofFloors invalide -> NaN + FloorsMissing : 16 lignes concernées (1.1% du jeu), conservées
Plus grand usage imputé (usage + surface) : 4 lignes concernées (0.3% du jeu), conservées
Surfaces d'usage incohérentes -> GFAInconsistent : 222 lignes concernées (14.7% du jeu), conservées
Colonnes constantes / vides supprimées : ['DataYear', 'City', 'State', 'DefaultData', 'Comments', 'ComplianceStatus', 'Outlier']
Seuils : 87,829 kBtu / 103,237,656 kBtu -> 12 outliers